In [ ]:
import os
import subprocess

REPO_DIR = "/content/kybelix-gid-segmentation"
if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/ysfztpp/kybelix-gid-segmentation.git", REPO_DIR],
        check=True,
    )

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "origin"], check=True)
subprocess.run(["git", "checkout", "main"], check=True)
subprocess.run(["git", "pull", "--ff-only", "origin", "main"], check=True)
print("Repo ready:", os.getcwd())


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path

# Toggle here:
# - drive_stream: old mode, reads directly from Drive folders.
# - zip_local_cache: copy zip from Drive to /content and train from local disk.
DATA_SOURCE_MODE = "zip_local_cache"
SHORTCUT_NAME = "phase1data"

os.environ["KYBELIX_DATA_SOURCE_MODE"] = DATA_SOURCE_MODE
os.environ["KYBELIX_SHORTCUT_NAME"] = SHORTCUT_NAME

if DATA_SOURCE_MODE == "zip_local_cache":
    drive_zip = Path(f"/content/drive/MyDrive/{SHORTCUT_NAME}.zip")
    local_zip = Path(f"/content/{SHORTCUT_NAME}.zip")
    local_root = Path(f"/content/{SHORTCUT_NAME}")

    if not drive_zip.exists():
        raise FileNotFoundError(f"Zip not found in Drive: {drive_zip}")

    needs_copy = (not local_zip.exists()) or (local_zip.stat().st_size != drive_zip.stat().st_size)
    if needs_copy:
        shutil.copy2(drive_zip, local_zip)
        print(f"Copied zip to: {local_zip}")
    else:
        print(f"Using existing local zip: {local_zip}")

    has_extracted = (local_root / "unmasked").is_dir() and (local_root / "masked").is_dir()
    if not has_extracted:
        with zipfile.ZipFile(local_zip, "r") as zf:
            zf.extractall("/content")
        print("Extracted zip into /content")
    else:
        print(f"Using existing extracted dataset: {local_root}")

    os.environ["KYBELIX_DATA_ROOT"] = str(local_root)
else:
    os.environ["KYBELIX_DATA_ROOT"] = f"/content/drive/MyDrive/{SHORTCUT_NAME}"

data_root = Path(os.environ["KYBELIX_DATA_ROOT"])
if not ((data_root / "unmasked").is_dir() and (data_root / "masked").is_dir()):
    raise FileNotFoundError(f"Dataset folders not found under: {data_root}")

print("KYBELIX_DATA_SOURCE_MODE =", os.environ["KYBELIX_DATA_SOURCE_MODE"])
print("KYBELIX_DATA_ROOT =", os.environ["KYBELIX_DATA_ROOT"])


In [ ]:
%cd /content/kybelix-gid-segmentation
!python3 -c "from config import Config; print('BASE_PATH:', Config.BASE_PATH); print('DATA_SOURCE_MODE:', Config.DATA_SOURCE_MODE)"


In [ ]:
%cd /content/kybelix-gid-segmentation
!python3 train.py


In [ ]:
# ====== Checkpoint verification ======
import os
from pathlib import Path
import torch

# Update this path to the checkpoint you want to use for inference
CKPT_PATH = "/content/drive/MyDrive/kybelix_runs/checkpoints/<RUN_NAME>/segformer_b4_best.pth"

if not os.path.isfile(CKPT_PATH):
    raise FileNotFoundError(f"Checkpoint not found: {CKPT_PATH}")

try:
    ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
except TypeError:
    ckpt = torch.load(CKPT_PATH, map_location="cpu")

print("Checkpoint path:", CKPT_PATH)
print("model_name:", ckpt.get("model_name"))
print("epoch:", ckpt.get("epoch"))
print("best_val_iou:", ckpt.get("best_val_iou"))
print("run_name:", ckpt.get("run_name"))

from config import Config
print("Config.MODEL_NAME:", Config.MODEL_NAME)
if ckpt.get("model_name") and ckpt.get("model_name") != Config.MODEL_NAME:
    print("WARNING: Config.MODEL_NAME and checkpoint model_name differ.")


In [ ]:
# ====== Test prediction + validation + visualization ======
import os
import subprocess
from config import Config

TEST_DIR = os.path.join(Config.BASE_PATH, "unmasked", "test")
OUT_DIR = "test_preds"
ZIP_NAME = "Team+Leader+email@example.com+1234567.zip"
REPORT_PATH = os.path.join(OUT_DIR, "validation_report.txt")
VERBOSE_VALIDATE = True

cmd = [
    "python3", "export_test_predictions.py",
    "--ckpt", CKPT_PATH,
    "--test-dir", TEST_DIR,
    "--out-dir", OUT_DIR,
    "--zip-name", ZIP_NAME,
    "--report", REPORT_PATH,
    "--viz-dir", "gorsel",
    "--viz-samples", "20",
]

if VERBOSE_VALIDATE:
    cmd.append("--verbose")

print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)
